# 07 — Climate Change Simulation

Simulate the effect of a +2°C temperature increase on crop yields across Turkey.

**Method:** Take the normal-scenario season data, add +2°C to `mean_temp`, run through the full-season model, compare predicted yields.

**Assumption:** All other features stay the same — this isolates the temperature effect.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from src.data.loader import load_season_agg

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('../data/processed')
OUTPUTS   = Path('../outputs')
FIGS      = '../outputs/figures/'

DELTA_T = 2.0  # °C

## 1. Load model and data

In [ ]:
model = joblib.load(OUTPUTS / 'models/baseline_lgbm_fullseason.joblib')
print('Model loaded.')

agg = load_season_agg()

# Normal scenario, successful harvests only
df = agg[(agg['wav_scenario'] == 'normal') & (agg['sim_success'] == 1)].copy()
print(f'Rows: {len(df):,}')

## 2. Target encoding

In [ ]:
with open(PROCESSED / 'crop_te_map.json') as f:
    crop_te_export = json.load(f)
global_mean = crop_te_export.pop('__global_mean__')
crop_te_map = crop_te_export

df['crop_te'] = df['crop_name'].map(crop_te_map).fillna(global_mean)

FEATURE_COLS = [
    'mean_temp', 'total_precip', 'mean_humidity', 'mean_rftra',
    'max_lai', 'max_tagp', 'max_dvs', 'season_days',
    'latitude', 'longitude', 'elevation', 'year', 'WAV', 'crop_te',
]

## 3. Predict — baseline vs +2°C

In [ ]:
X_base = df[FEATURE_COLS].copy()
X_warm = X_base.copy()
X_warm['mean_temp'] += DELTA_T

df['yield_base'] = model.predict(X_base)
df['yield_warm'] = model.predict(X_warm)
df['yield_change']    = df['yield_warm'] - df['yield_base']
df['yield_change_pct'] = df['yield_change'] / df['yield_base'] * 100

print(f'Mean yield change: {df["yield_change"].mean():+.1f} kg/ha')
print(f'Mean yield change: {df["yield_change_pct"].mean():+.2f}%')

## 4. Effect by crop

In [ ]:
crop_impact = (
    df.groupby('crop_name')
    .agg(yield_base=('yield_base', 'median'),
         yield_warm=('yield_warm', 'median'),
         change_pct=('yield_change_pct', 'median'))
    .sort_values('change_pct')
)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in crop_impact['change_pct']]
ax.barh(crop_impact.index, crop_impact['change_pct'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Median Yield Change (%)')
ax.set_title(f'Yield Change with +{DELTA_T}°C — by Crop (Normal Scenario)')
plt.tight_layout()
plt.savefig(FIGS + 'climate_change_by_crop.png', bbox_inches='tight')
plt.show()

print(crop_impact[['yield_base', 'yield_warm', 'change_pct']].to_string())

## 5. Spatial effect — map

In [ ]:
# Average across all crops and years per location
spatial = (
    df.groupby(['latitude', 'longitude'])
    ['yield_change_pct'].median().reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
sc = ax.scatter(spatial['longitude'], spatial['latitude'],
                c=spatial['yield_change_pct'],
                cmap='RdYlGn', s=200, marker='s',
                edgecolors='gray', linewidths=0.3,
                vmin=-20, vmax=20)
plt.colorbar(sc, ax=ax, label='Median yield change (%)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Spatial Yield Impact of +{DELTA_T}°C Warming (All Crops, Normal Scenario)')
plt.tight_layout()
plt.savefig(FIGS + 'climate_change_spatial.png', bbox_inches='tight')
plt.show()

## 6. Wheat deep-dive

In [ ]:
wheat = df[df['crop_name'] == 'wheat'].copy()

# Yield change by year
yearly = wheat.groupby('year')['yield_change_pct'].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Year trend
axes[0].plot(yearly.index, yearly.values, marker='o', color='steelblue')
axes[0].axhline(0, ls='--', color='red', lw=1)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Median yield change (%)')
axes[0].set_title(f'Wheat: +{DELTA_T}°C Effect by Year')

# Spatial
wheat_spatial = wheat.groupby(['latitude', 'longitude'])['yield_change_pct'].median().reset_index()
sc = axes[1].scatter(wheat_spatial['longitude'], wheat_spatial['latitude'],
                     c=wheat_spatial['yield_change_pct'],
                     cmap='RdYlGn', s=200, marker='s',
                     edgecolors='gray', linewidths=0.3,
                     vmin=-20, vmax=20)
plt.colorbar(sc, ax=axes[1], label='Yield change (%)')
axes[1].set_title(f'Wheat: Spatial Yield Impact of +{DELTA_T}°C')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')

plt.tight_layout()
plt.savefig(FIGS + 'climate_change_wheat.png', bbox_inches='tight')
plt.show()

## 7. Save

In [ ]:
df[['latitude', 'longitude', 'crop_name', 'year',
    'yield_base', 'yield_warm', 'yield_change', 'yield_change_pct']].to_parquet(
    OUTPUTS / 'climate_change_simulation.parquet', index=False
)

summary = {
    'delta_temp_c': DELTA_T,
    'mean_yield_change_pct': round(float(df['yield_change_pct'].mean()), 2),
    'median_yield_change_pct': round(float(df['yield_change_pct'].median()), 2),
    'most_affected_crop': crop_impact['change_pct'].idxmin(),
    'least_affected_crop': crop_impact['change_pct'].idxmax(),
}
with open(OUTPUTS / 'climate_change_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved.')
print(json.dumps(summary, indent=2))